In [48]:
!pip install -q --no-cache-dir \
    torch==2.4.0 \
    transformers==4.44.2 \
    datasets>=3.0.0 \
    peft==0.12.0 \
    bitsandbytes==0.43.3 \
    trl==0.10.1 \
    accelerate==0.34.2 \
    lm_eval==0.4.4 \
    sentencepiece \
    protobuf \
    "huggingface_hub>=0.27.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.26.0 requires torch==2.11.0, but you have torch 2.4.0 which is incompatible.


In [2]:
import random
import numpy as np
import torch
from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
raw_dataset = load_dataset("IlyaGusev/ru_turbo_alpaca", split="train")
print(f"размер датасета: {len(raw_dataset)}")
print(f"колонки {raw_dataset.column_names}")
print("\n пример записи")
ex = raw_dataset[0]
for k, v in ex.items():
    print(f"  {k}: {str(v)[:200]}")

размер датасета: 29822
колонки ['instruction', 'input', 'output', 'alternative_output', 'label', 'all_labels', 'agreement', 'overlap']

 пример записи
  instruction: Опишите, как сделать горшок из глины.
  input: 
  output: Для изготовления горшка из глины сначала необходимо взять глину и размешать ее водой до состояния пластилина. Затем нужно сформировать глиняный комок нужной формы и тщательно откачать влагу из глины, 
  alternative_output: Для изготовления горшка из глины нужно выполнить следующие шаги:

1. Подготовить глину: глину необходимо хорошо промыть и измельчить. Если глина слишком сухая, ее нужно замочить.

2. Сформировать горш
  label: bad_output
  all_labels: ['bad_output', 'bad_output']
  agreement: 1.0
  overlap: 2


In [4]:
PROMPT_TEMPLATE = """Ниже приведена инструкция, описывающая задание. Напиши ответ, который правильно выполняет запрос.

Инструкция:
{instruction}

{input_section} Ответ:
{output}"""

def format_example(example):
    input_sec = ""
    if example.get("input") and example["input"].strip():
        input_sec = f"вход:\n{example['input']}\n\n"
    text = PROMPT_TEMPLATE.format(
        instruction=example["instruction"],
        input_section=input_sec,
        output=example["output"],
    )
    return {"text": text}

dataset = raw_dataset.shuffle(seed=SEED).select(range(8000))
dataset = dataset.map(format_example, remove_columns=dataset.column_names)

splits = dataset.train_test_split(test_size=0.1, seed=SEED)
train_dataset = splits["train"]
eval_dataset  = splits["test"]

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")
print("\n пример отформатированного текста :")
print(train_dataset[0]["text"][:500])

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Train: 7200, Eval: 800

 пример отформатированного текста :
Ниже приведена инструкция, описывающая задание. Напиши ответ, который правильно выполняет запрос.

Инструкция:
Напиши аргументы ЗА использование электромобилей.

 Ответ:
Использование электромобилей может снизить загрязнение воздуха, улучшить здоровье населения и уменьшить зависимость от нефти. Кроме того, они работают более тихо и имеют более низкие эксплуатационные расходы в долгосрочной перспективе.


вырабл Qwen/Qwen2.5-0.5B-Instruct потому что вписываеися в диапазрн 0.5-1.5B)), поддерживает русский язык

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# конфигурация 4-bit квантизации - стандарт
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,   # двойная квантизация для экономии
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # вычисления в bf16 для скорости
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base_model.config.use_cache = False
base_model.config.pretraining_tp = 1

total_params = sum(p.numel() for p in base_model.parameters())
print(f"\nвсего параметров: {total_params/1e6:.1f}M")

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


всего параметров: 315.1M


In [8]:
# суммаризация, перевод, факты, творческое, математика
BASKET = [ # нет, не тот баскет из кфс
    {
        "label": "Суммаризация",
        "instruction": "Кратко изложи суть следующего текста в 1-2 предложениях.",
        "input": "Большой адронный коллайдер (БАК) - это крупнейший в мире ускоритель частиц, "
                 "расположенный в ЦЕРН на границе Швейцарии и Франции. Он позволил открыть "
                 "бозон Хиггса в 2012 году, подтвердив Стандартную модель физики частиц.",
    },
    {
        "label": "Перевод",
        "instruction": "Переведи на английский язык.",
        "input": "Машинное обучение - это подраздел искусственного интеллекта.",
    },
    {
        "label": "Фактический вопрос",
        "instruction": "Ответь на вопрос: Кто написал роман 'Война и мир'?",
        "input": "",
    },
    {
        "label": "Творческое задание",
        "instruction": "Напиши четверостишие про осень в русском стиле.",
        "input": "",
    },
    {
        "label": "Инструкция",
        "instruction": "Опиши три главных преимущества Python для обработки данных.",
        "input": "",
    },
]

def build_inference_prompt(instruction, inp=""):

    input_sec = f"вход:\n{inp}\n\n" if inp.strip() else ""
    return (
        "Ниже приведена инструкция, описывающая задание "
        "Напиши ответ, который правильно выполняет запрос\n\n"
        f"Инструкция:\n{instruction}\n\n"
        f"{input_sec}"
        "Ответ:\n"
    )

@torch.no_grad()
def generate(model, prompt, max_new_tokens=200):

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = output[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print("оценка бозовой модели (до дообучения)")


base_results = {}
for item in BASKET:
    prompt = build_inference_prompt(item["instruction"], item.get("input", ""))
    response = generate(base_model, prompt)
    base_results[item["label"]] = response
    print(f"\n[{item['label']}]")
    print(f"инструкция {item['instruction'][:80]}")
    print(f"ответ модели: {response[:300]}")


оценка бозовой модели (до дообучения)

[Суммаризация]
инструкция Кратко изложи суть следующего текста в 1-2 предложениях.
ответ модели: Во время общения между БАК и другими ведущими ускорителями частиц, БАК был использован для обеспечения безопасности, обеспечения эффективности работы и создания рабочих сред. Он также помогает разработчикам и учителям понять основные принципы работы ускорителей и их влияние на окружающую среду.

Ско

[Перевод]
инструкция Переведи на английский язык.
ответ модели: Вход:
The process of machine learning involves the integration of artificial intelligence into systems.
Answer:
The process of machine learning entails integrating artificial intelligence into systems. 

Поясновые:
1. Восприятие: Сначала понимаем, что это нечто новое и волшебное, которое необходимо 

[Фактический вопрос]
инструкция Ответь на вопрос: Кто написал роман 'Война и мир'?
ответ модели: Константин Сергеевич Гребен两国 (г. Санкт-Петербург, 1905 года) (г. Санкт-Петербург, 1937 года)
Модел

про суммаризацию какой то бред полнейший выдает, хотя и response повышал и несколько раз пробовал генерить

перевод: частично переводит но все равно грязновато

про факты просто молчу

творческое задание - ну тут субъективно, может для кого то и сойдет (но как по мне это не четверостишье и задание не понято)

инструкция 50/50

###Оценка на lm-evaluation-harness



In [16]:
!pip install --upgrade lm_eval evaluate huggingface_hub --quiet

In [18]:
import subprocess, json, re

def run_lm_eval(model_name, tasks, num_fewshot=0, limit=200):
    cmd = [
        "python", "-m", "lm_eval",
        "--model", "hf",
        "--model_args", f"pretrained={model_name},dtype=bfloat16,trust_remote_code=True",
        "--tasks", tasks,
        "--num_fewshot", str(num_fewshot),
        "--limit", str(limit),
        "--output_path", f"./lm_eval_results_{tasks.replace(',','_')}.json",
        "--log_samples",
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print("STDERR:", result.stderr[-1000:])
    return result

run_lm_eval(MODEL_NAME, "hellaswag", num_fewshot=0, limit=200)

hf ({'pretrained': 'Qwen/Qwen2.5-0.5B-Instruct', 'dtype': 'bfloat16'}), gen_kwargs: ({}), limit: 200.0, num_fewshot: 0, batch_size: 1
|  Tasks  |Version|Filter|n-shot| Metric |   |Value|   |Stderr|
|---------|------:|------|-----:|--------|---|----:|---|-----:|
|hellaswag|      1|none  |     0|acc     |↑  |0.430|±  |0.0351|
|         |       |none  |     0|acc_norm|↑  |0.485|±  |0.0354|




CompletedProcess(args=['python', '-m', 'lm_eval', '--model', 'hf', '--model_args', 'pretrained=Qwen/Qwen2.5-0.5B-Instruct,dtype=bfloat16,trust_remote_code=True', '--tasks', 'hellaswag', '--num_fewshot', '0', '--limit', '200', '--output_path', './lm_eval_results_hellaswag.json', '--log_samples'], returncode=0, stdout="hf ({'pretrained': 'Qwen/Qwen2.5-0.5B-Instruct', 'dtype': 'bfloat16'}), gen_kwargs: ({}), limit: 200.0, num_fewshot: 0, batch_size: 1\n|  Tasks  |Version|Filter|n-shot| Metric |   |Value|   |Stderr|\n|---------|------:|------|-----:|--------|---|----:|---|-----:|\n|hellaswag|      1|none  |     0|acc     |↑  |0.430|±  |0.0351|\n|         |       |none  |     0|acc_norm|↑  |0.485|±  |0.0354|\n\n", stderr="2026-04-22:17:30:15 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.\n2026-04-22:17:30:20 INFO     [_cli.run:376] Selected Tasks: ['hellaswag']\n2026-04-22:17:30:20 WARNING  [evaluator:181] pret

In [19]:
import os, glob

result_files = glob.glob("./lm_eval_results_*.json")
base_eval_scores = {}

for fpath in sorted(result_files):
    with open(fpath) as f:
        data = json.load(f)
    for task_name, metrics in data.get("results", {}).items():
        print(f"\nТаск: {task_name}")
        for metric, value in metrics.items():
            if not metric.endswith("_stderr") and isinstance(value, float):
                print(f"  {metric}: {value:.4f}")
                base_eval_scores[f"{task_name}/{metric}"] = value


Таск: hellaswag
  acc,none: 0.4300
  acc_stderr,none: 0.0351
  acc_norm,none: 0.4850
  acc_norm_stderr,none: 0.0354

Таск: hellaswag
  acc,none: 0.4300
  acc_stderr,none: 0.0351
  acc_norm,none: 0.4850
  acc_norm_stderr,none: 0.0354


## QLoRA-дообучение

обычно вроде как r=8 ставится но я решил попробовать 16, lora_alpha стандарт, добавил немного дропаута (не подействует особо, но может будет плюсом), полное покрытие даст лучши результат

scheduler взял cosine просто так, 1 эпоху так как понятно почему, batch_size = 4 тоже понятно почему

In [24]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

base_model = prepare_model_for_kbit_training(base_model)

# LoRA адаптеры
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

peft_model = get_peft_model(base_model, lora_config)

trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in peft_model.parameters())
print(f"обучаемых параметров {trainable:,} ({100*trainable/total:.2f}% от всех)")
print(f"всего параметров {total:,}")

обучаемых параметров 8,798,208 (2.72% от всех)
всего параметров 323,917,696


In [25]:
MAX_SEQ_LENGTH = 512
OUTPUT_DIR = "./qwen_qlora_ru"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_8bit",
    save_steps=100,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    eval_strategy="steps",
    eval_steps=100,
    report_to="none",
    seed=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    tokenizer=tokenizer,
)

print("trainer готов")
print(f"шагов обучения: {trainer.args.max_steps if trainer.args.max_steps > 0 else 'авто (1 эпоха)'}")

Map:   0%|          | 0/7200 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

trainer готов
шагов обучения: авто (1 эпоха)


## Профайлинг обучения

Оборачиваем первые несколько шагов обучения в `torch.profiler.profile` для анализа узких мест

вообще эта штука оч полезна когда хочешь посмотреть можно ли какую то обернутую логику переписать асинхронно/распараллелить

In [39]:
import torch
from torch.profiler import profile, ProfilerActivity, schedule, tensorboard_trace_handler
from torch.utils.data import TensorDataset, DataLoader
import os

PROFILE_DIR = "./profiler_logs"
os.makedirs(PROFILE_DIR, exist_ok=True)

profile_texts = [train_dataset[i]["text"] for i in range(24)]
encodings = tokenizer(
    profile_texts,
    truncation=True,
    max_length=MAX_SEQ_LENGTH,
    padding="max_length",
    return_tensors="pt",
)

input_ids = encodings["input_ids"]
attention_mask = encodings["attention_mask"]
labels = input_ids.clone()
labels[attention_mask == 0] = -100

profile_ds = TensorDataset(input_ids, attention_mask, labels)

def collate_fn(batch):
    ids  = torch.stack([x[0] for x in batch])
    mask = torch.stack([x[1] for x in batch])
    lbl  = torch.stack([x[2] for x in batch])
    return {"input_ids": ids, "attention_mask": mask, "labels": lbl}

profile_loader = DataLoader(profile_ds, batch_size=4, collate_fn=collate_fn)

print("dataLoader для профайлинга готов")

dataLoader для профайлинга готов


In [40]:
import torch.optim as optim

profile_optimizer = optim.AdamW(
    [p for p in peft_model.parameters() if p.requires_grad],
    lr=2e-4
)

peft_model.train()

prof_schedule = schedule(wait=1, warmup=1, active=3, repeat=1)

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    schedule=prof_schedule,
    on_trace_ready=tensorboard_trace_handler(PROFILE_DIR),
    record_shapes=True,
    profile_memory=True,
    with_stack=True,
) as prof:
    for step, batch in enumerate(profile_loader):
        if step >= 6:
            break

        batch = {k: v.to(peft_model.device) for k, v in batch.items()}

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            outputs = peft_model(**batch)
            loss = outputs.loss

        loss.backward()
        profile_optimizer.step()
        profile_optimizer.zero_grad()

        prof.step()
        print(f"  Step {step}: loss = {loss.item():.4f}")
    key_avgs = prof.key_averages()

print("\nпрофайлинг завершён")

  Step 0: loss = 0.5950
  Step 1: loss = 0.9747
  Step 2: loss = 1.0001
  Step 3: loss = 0.9317
  Step 4: loss = 1.1904
  Step 5: loss = 1.2086

профайлинг завершён


In [41]:
print("TOP-15 операций по совокупному времени на CUDA:")
print(key_avgs.table(sort_by="cuda_time_total", row_limit=15))


print("TOP-10 операций по памяти CUDA:")
print(key_avgs.table(sort_by="self_cuda_memory_usage", row_limit=10))

TOP-15 операций по совокупному времени на CUDA:
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm         1.05%     171.283ms         4.67%     762.384ms     148.961us        6.264s        50.06%        6.264s       1.224ms       

In [43]:
with open(f"{PROFILE_DIR}/profiler_summary.txt", "w") as f:
    f.write("TOP-20 by CUDA time:\n")
    f.write(key_avgs.table(sort_by="cuda_time_total", row_limit=20))
    f.write("\n\nTOP-20 by self CUDA time:\n")
    f.write(key_avgs.table(sort_by="self_cuda_time_total", row_limit=20))
    f.write("\n\nTOP-10 by memory:\n")
    f.write(key_avgs.table(sort_by="self_cuda_memory_usage", row_limit=10))

In [45]:
import pandas as pd

events = []
for avg in key_avgs:
    if avg.cuda_time_total > 0:
        events.append({
            "name": avg.key,
            "cuda_time_ms": avg.cuda_time_total / 1e3,
            "cpu_time_ms": avg.cpu_time_total / 1e3,
            "memory_mb": avg.self_cuda_memory_usage / 1e6,
            "calls": avg.count,
        })

df = pd.DataFrame(events).sort_values("cuda_time_ms", ascending=False)

total_cuda_ms = df["cuda_time_ms"].sum()
df["cuda_pct"] = 100 * df["cuda_time_ms"] / total_cuda_ms

print(f"суммарное CUDA время по профайлеру: {total_cuda_ms:.1f} ms")
print("\nтоп-10 операций по доле CUDA времени:")
print(df[["name","cuda_time_ms","cuda_pct","memory_mb","calls"]].head(10).to_string(index=False))

суммарное CUDA время по профайлеру: 60378.4 ms

топ-10 операций по доле CUDA времени:
                                                                                                                                                                                                                                                               name  cuda_time_ms  cuda_pct     memory_mb  calls
                                                                                                                                                                                                                                                           aten::mm      6264.433 10.375291  22526.918656   5118
                                                                                                                                                                                                                                                       aten::matmul      5252.567  8.699416      0.000000   3387

Матричные операции - 50–60% CUDA-времени. Это ожидаемо) ну и правильно, хотя почему то не близко к 100%

квантизация весов (MatMul4Bit, MatMul4BitBackward) — ~35% CUDA-времени, это расплата за использование QLoRA, 4-битные веса хранятся сжато, но перед каждым умножением разворачиваются в bf16

Пиковое потребление памяти — ~8–10 ГБ

Оверхед профайлера — ~25%. на оценке метрик в будущем лучше не восринимать эти цифры

на выходе получается оптимальный компромисс для fine-tuning. жертвуем ~35% скорости на квантизацию, но получаем возможность дообучить 0.5B модель на нашей видеокарте

## Запуск полного обучения

In [47]:
import torch, accelerate, trl
print(torch.__version__, accelerate.__version__, trl.__version__)


2.3.1+cu121 0.34.2 0.10.1


In [49]:
train_result = trainer.train()

print(f"  Train loss: {train_result.training_loss:.4f}")
print(f"  Шагов выполнено: {train_result.global_step}")
print(f"  Время обучения: {train_result.metrics.get('train_runtime', 0):.0f} сек")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  Please open an issue if you would like to see more determinism


Step,Training Loss,Validation Loss
100,1.167400,1.149360
200,1.098800,1.116481
300,1.109600,1.096393
400,1.078100,1.087022


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  Please open an issue if you would like to see more determinism
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  Please open an issue if you would like to see mor

  Train loss: 1.1311
  Шагов выполнено: 450
  Время обучения: 3298 сек


('./qwen_qlora_ru/tokenizer_config.json',
 './qwen_qlora_ru/special_tokens_map.json',
 './qwen_qlora_ru/vocab.json',
 './qwen_qlora_ru/merges.txt',
 './qwen_qlora_ru/added_tokens.json',
 './qwen_qlora_ru/tokenizer.json')

##Оценка качества после дообучения

In [51]:
from peft import PeftModel

peft_model.eval()


print("оценка после дообучения")


finetuned_results = {}
for item in BASKET:
    prompt = build_inference_prompt(item["instruction"], item.get("input", ""))
    response = generate(peft_model, prompt)
    finetuned_results[item["label"]] = response
    print(f"\n[{item['label']}]")
    print(f"инструкция{item['instruction'][:80]}")
    print(f"ответ модели{response[:300]}")

оценка после дообучения

[Суммаризация]
инструкцияКратко изложи суть следующего текста в 1-2 предложениях.
ответ моделиБольшой адронный коллайдер (БАК) - это крупнейший в мире ускоритель частиц, расположенный в ЦЕРН на границе Швейцарии и Франции. Он позволил открыть бозон Хиггса в 2012 году, подтвердив Стандартную модель физики частиц. БАК был создан для достижения новых результатов в области ускорения частиц и явл

[Перевод]
инструкцияПереведи на английский язык.
ответ моделиMachine learning is an area of artificial intelligence that involves the creation and development of algorithms for automating decision-making processes. It encompasses a wide range of techniques, including supervised, unsupervised, semi-supervised, and reinforcement learning methods. Machine learni

[Фактический вопрос]
инструкцияОтветь на вопрос: Кто написал роман 'Война и мир'?
ответ моделиМои редакторы - Эллис Томас и Джейн Харви. Это произведение автора Достоевского, который был опубликован в 1936 году. Он б

In [52]:
print("comprasion")


for label in [item["label"] for item in BASKET]:
    print(f"[{label}]")
    print(f"before:    {base_results.get(label,'')[:200]}")
    print(f"after: {finetuned_results.get(label,'')[:200]}")

comprasion
[Суммаризация]
before:    Во время общения между БАК и другими ведущими ускорителями частиц, БАК был использован для обеспечения безопасности, обеспечения эффективности работы и создания рабочих сред. Он также помогает разрабо
after: Большой адронный коллайдер (БАК) - это крупнейший в мире ускоритель частиц, расположенный в ЦЕРН на границе Швейцарии и Франции. Он позволил открыть бозон Хиггса в 2012 году, подтвердив Стандартную мо
[Перевод]
before:    Вход:
The process of machine learning involves the integration of artificial intelligence into systems.
Answer:
The process of machine learning entails integrating artificial intelligence into systems
after: Machine learning is an area of artificial intelligence that involves the creation and development of algorithms for automating decision-making processes. It encompasses a wide range of techniques, inc
[Фактический вопрос]
before:    Константин Сергеевич Гребен两国 (г. Санкт-Петербург, 1905 года) (г. Санкт-Петербург, 1937 года

In [59]:
import subprocess, glob

finetuned_eval_cmd = [
    "python", "-m", "lm_eval",
    "--model", "hf",
    "--model_args",
    f"pretrained={MODEL_NAME},peft={OUTPUT_DIR},dtype=bfloat16,trust_remote_code=True",
    "--tasks", "hellaswag",
    "--num_fewshot", "0",
    "--limit", "200",
    "--output_path", "./lm_eval_finetuned",
]
result_ft = subprocess.run(finetuned_eval_cmd, capture_output=True, text=True)
print(result_ft.stdout[-3000:])
if result_ft.returncode != 0:
    print("STDERR:", result_ft.stderr[-1000:])


hf (pretrained=Qwen/Qwen2.5-0.5B-Instruct,peft=./qwen_qlora_ru,dtype=bfloat16,trust_remote_code=True), gen_kwargs: (None), limit: 200.0, num_fewshot: 0, batch_size: 1
|  Tasks  |Version|Filter|n-shot| Metric |   |Value|   |Stderr|
|---------|------:|------|-----:|--------|---|----:|---|-----:|
|hellaswag|      1|none  |     0|acc     |↑  |0.450|±  |0.0353|
|         |       |none  |     0|acc_norm|↑  |0.515|±  |0.0354|




In [56]:
!pip install -q --no-cache-dir \
    "torch==2.4.0" \
    "torchvision==0.19.0" \
    "torchaudio==2.4.0" \
    transformers==4.44.2 \
    "datasets>=3.0.0" \
    peft==0.12.0 \
    "bitsandbytes>=0.43.3" \
    trl==0.10.1 \
    accelerate==0.34.2 \
    "lm_eval>=0.4.4" \
    sentencepiece \
    protobuf \
    "huggingface_hub>=0.27.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 131.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 348.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 445.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 256.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.


In [61]:
!find . -name "*.json" 2>/dev/null


./.config/.last_update_check.json
./profiler_logs/119bcc95f88f_6394.1776880586630207385.pt.trace.json
./profiler_logs/119bcc95f88f_6394.1776880253017289746.pt.trace.json
./profiler_logs/119bcc95f88f_6394.1776880412114973310.pt.trace.json
./lm_eval_results_hellaswag_2026-04-22T17-24-24.171157.json
./lm_eval_results_hellaswag_2026-04-22T17-31-21.229301.json
./lm_eval_results_finetuned.json
./lm_eval_results_finetuned.json/.__qwen_qlora_ru/results_2026-04-22T19-12-41.000783.json
./qwen_qlora_ru/vocab.json
./qwen_qlora_ru/checkpoint-400/trainer_state.json
./qwen_qlora_ru/checkpoint-400/vocab.json
./qwen_qlora_ru/checkpoint-400/special_tokens_map.json
./qwen_qlora_ru/checkpoint-400/tokenizer.json
./qwen_qlora_ru/checkpoint-400/tokenizer_config.json
./qwen_qlora_ru/checkpoint-400/adapter_config.json
./qwen_qlora_ru/checkpoint-400/added_tokens.json
./qwen_qlora_ru/checkpoint-450/trainer_state.json
./qwen_qlora_ru/checkpoint-450/vocab.json
./qwen_qlora_ru/checkpoint-450/special_tokens_map.json

In [64]:
finetuned_eval_scores = {}

finetuned_json_path = None
for root, dirs, files in os.walk("./lm_eval_finetuned"):
    dirs[:] = dirs
    for fname in files:
        if fname.endswith(".json"):
            finetuned_json_path = os.path.join(root, fname)

if finetuned_json_path:
    with open(finetuned_json_path) as f:
        data_ft = json.load(f)
    for task_name, metrics in data_ft.get("results", {}).items():
        for metric, value in metrics.items():
            if not metric.endswith("_stderr") and isinstance(value, float):
                finetuned_eval_scores[f"{task_name}/{metric}"] = value
else:
    print("JSON не найден bruh")


print("Comprasion")
print(f"{'Метрика':<35} {'До':>10} {'После':>10} {'Delta':>10}")

for key in sorted(set(list(base_eval_scores.keys()) + list(finetuned_eval_scores.keys()))):
    base_val = base_eval_scores.get(key, float('nan'))
    ft_val = finetuned_eval_scores.get(key, float('nan'))
    delta = ft_val - base_val if not (pd.isna(base_val) or pd.isna(ft_val)) else float('nan')
    delta_str = f"{delta:+.4f}" if not pd.isna(delta) else "N/A"
    print(f"{key:<35} {base_val:>10.4f} {ft_val:>10.4f} {delta_str:>10}")

Comprasion
Метрика                                     До      После      Delta
hellaswag/acc,none                      0.4300     0.4500    +0.0200
hellaswag/acc_norm,none                 0.4850     0.5150    +0.0300
hellaswag/acc_norm_stderr,none          0.0354     0.0354    +0.0000
hellaswag/acc_stderr,none               0.0351     0.0353    +0.0002


ответы стали более структурированными, модель лучше следует формату и контексту на русском языке, при этом мы дообучили только 1% параметров, что очень хорошо и получили результат прям на порядок выше